In [1]:
from readability import Readability
import nltk

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
import pandas as pd

In [3]:
dataset = 'dev_small'
retriever = 'e5'
retr_res = pd.read_csv(f'../../rag_utility/res/{retriever}_{dataset}.csv')

In [4]:
import pyterrier as pt

if not pt.java.started():
    pt.java.init()

Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [5]:
if('nq' in dataset):
    index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')
else:
    index_path ='/mnt/indices/msmarco-passage.terrier/'
    index_ref = pt.IndexRef.of(index_path)
    index = pt.IndexFactory.of(index_ref)

# index_path ='/mnt/indices/msmarco-passage.terrier/'
# index_ref = pt.IndexRef.of(index_path)
# index = pt.IndexFactory.of(index_ref)

15:31:53.038 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1.9 GiB of memory would be required.


In [6]:
text_loader = index.text_loader(["text"])

In [7]:
def calculate_readability(_input_text: str, _qid: str, _docno: str):
    r = Readability(_input_text)
    # Map names to the corresponding method calls (lambdas delay execution)
    metrics = {
        "Dale Chall": lambda: r.dale_chall(),
        "Spache": lambda: r.spache(),
        "Flesch-Kincaid": lambda: r.flesch_kincaid(),
        "Flesch": lambda: r.flesch(),
        "Gunning Fog": lambda: r.gunning_fog(),
        "Coleman Liau": lambda: r.coleman_liau(),
        "ARI": lambda: r.ari(),
        "Linsear Write": lambda: r.linsear_write(),
        "SMOG": lambda: r.smog(),
    }
    
    to_output = []
    for name, func in metrics.items():
        try:
            readability_result = func()
            score = readability_result.score
            try:
                grade_level = readability_result.grade_levels
            except:
                grade_level = []
            to_output.append([_qid, _docno, name, score, grade_level])
        except Exception as e:
            to_output.append([_qid, _docno, name, -1, []])
            # print(f"No {name}: {e}")

    # if(len(to_output)==0):
    #     print(_qid)
    return to_output

In [8]:
from tqdm import tqdm
import pathlib

_k = 2

output_path = f"./readability_res/integrated_readability_{dataset}_{retriever}_top_{_k}.csv"
try:
    exist_qids = pd.read_csv(output_path).qid.unique().values
except:
    exist_qids = []

for qid in tqdm(retr_res.qid.unique()):
    if(qid in exist_qids):
        continue
    df_content = []
    input = retr_res[(retr_res.qid==qid)&(retr_res['rank']<_k)]
    input_text = ''
    docno = f'{qid}_integrated_{_k}'
    
    for _t in text_loader(input).text.values:
        input_text += _t
    
    df_content += calculate_readability(input_text, qid, docno)


    temp_output = pd.DataFrame(df_content, columns=['qid', 'docno', 'readability_metric', 'score', 'grade'])

    csvfile = pathlib.Path(output_path)
    temp_output.to_csv(output_path, mode='a', index=False, header=not csvfile.exists())
    # print(temp_output)

100%|██████████| 6980/6980 [03:22<00:00, 34.55it/s]
